# CodeBERT Vulnerability Detection Training

Run this entire notebook to train a vulnerability detection model on your dataset using Google Colab's free GPU.

### Instructions:
1. In the menu, go to **Runtime > Change runtime type** and select **T4 GPU** (or A100 if you have Colab Pro).
2. Run the cells below in order.
3. **IMPORTANT**: If the first cell asks you to 'Restart Session', click that button, then run the cell again!

In [ ]:
# 1. Install Dependencies
!pip install transformers[torch] datasets accelerate scikit-learn tensorboard --upgrade
# NOTE: If you see a 'Restart Session' button below, click it to update dependencies!

In [ ]:
# 2. Download Dataset
import os

if os.path.exists('diversevul_20230702.json'):
    print("\u2705 Dataset already exists!")
else:
    print("--- Downloading Dataset from Google Drive Link ---")
    !gdown --id 12IWKhmLhq7qn5B_iXgn5YerOQtkH-6RG

In [ ]:
# 3. Preprocess Data (Dynamic Padding)
import json
from sklearn.model_selection import train_test_split
from transformers import RobertaTokenizer
from datasets import Dataset
from tqdm.auto import tqdm

print("Loading dataset...")
records = []
try:
    with open("diversevul_20230702.json", "r", encoding="utf-8") as f:
        for line in tqdm(f):
            line = line.strip()
            if not line: continue
            d = json.loads(line)
            if d.get("func") and len(d["func"]) <= 8000:
                records.append({"func": d["func"], "label": int(d["target"])})
except FileNotFoundError:
    print("\u274c Error: diversevul_20230702.json not found! Download might have failed.")
    raise

print(f"Loaded {len(records)} records.")

# 80/10/10 Split
labels = [r["label"] for r in records]
try:
    train_val, test = train_test_split(records, test_size=0.1, stratify=labels, random_state=42)
    train_val_labels = [r["label"] for r in train_val]
    train, val = train_test_split(train_val, test_size=0.1111, stratify=train_val_labels, random_state=42)
except ValueError:
    train_val, test = train_test_split(records, test_size=0.1, random_state=42)
    train, val = train_test_split(train_val, test_size=0.1111, random_state=42)

print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

# Tokenize — NO static padding! Dynamic padding happens per-batch at train time.
tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")
def tokenize_batch(batch):
    return tokenizer(batch["func"], truncation=True, padding=False, max_length=512)

for split_name, data in [("train", train), ("val", val), ("test", test)]:
    print(f"Tokenizing {split_name}...")
    ds = Dataset.from_list(data)
    ds = ds.map(tokenize_batch, batched=True, batch_size=1000, remove_columns=["func"])
    ds.save_to_disk(f"data/{split_name}")

print("\u2705 Preprocessing complete!")

In [ ]:
# 4. Train Model (Focal Loss + Dynamic Padding + Speed Optimizations)
import torch
import numpy as np
import random
from datasets import load_from_disk
from transformers import (
    RobertaForSequenceClassification,
    RobertaTokenizer,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB)")

if device.type == 'cpu':
    print("\u274c WARNING: You are using CPU! Training will be extremely slow.")
    print("Please go to Runtime > Change runtime type > Select T4 GPU.")

train_ds = load_from_disk("data/train")
val_ds = load_from_disk("data/val")

# =====================================================
#  Undersample majority class to 3:1 ratio
# =====================================================
random.seed(42)
train_labels = train_ds["label"]
pos_indices = [i for i, l in enumerate(train_labels) if l == 1]
neg_indices = [i for i, l in enumerate(train_labels) if l == 0]

target_neg_count = len(pos_indices) * 3
sampled_neg = random.sample(neg_indices, min(target_neg_count, len(neg_indices)))
balanced_indices = pos_indices + sampled_neg
random.shuffle(balanced_indices)

train_ds_balanced = train_ds.select(balanced_indices)
num_pos = len(pos_indices)
num_neg = len(sampled_neg)
print(f"\n\u2705 Balanced training set: {len(train_ds_balanced)} samples")
print(f"   Positive (vulnerable): {num_pos}")
print(f"   Negative (safe):       {num_neg}")
print(f"   Ratio:                 1:{num_neg/num_pos:.1f}")

model = RobertaForSequenceClassification.from_pretrained("microsoft/codebert-base", num_labels=2)
model.to(device)

# Dynamic padding — pads each batch to its longest sample, not to 512
tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)

# =====================================================
#  Focal Loss — focuses training on hard examples
#  alpha=5.0 gives vulnerable samples 5x weight
#  gamma=2.0 down-weights easy-to-classify samples
# =====================================================
class FocalLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        alpha = torch.tensor([1.0, 5.0]).to(logits.device)
        gamma = 2.0

        ce_loss = torch.nn.functional.cross_entropy(logits, labels, weight=alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** gamma * ce_loss).mean()

        return (focal_loss, outputs) if return_outputs else focal_loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    try: auc = roc_auc_score(labels, probs)
    except: auc = 0.5
    return {"accuracy": acc, "f1": f1, "recall": recall, "precision": precision, "auc": auc}

training_args = TrainingArguments(
    output_dir="model_output",
    num_train_epochs=4,                          # 4 epochs (was 10)
    per_device_train_batch_size=32,              # 32 (was 16) — T4 handles this fine
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=2,               # Effective batch = 64
    eval_strategy="steps",
    eval_steps=500,                              # Eval less often (was 200)
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    learning_rate=2e-5,                          # Lower LR for stability (was 5e-5)
    weight_decay=0.01,
    warmup_steps=200,                            # Fixed warmup steps (was warmup_ratio)
    fp16=True if torch.cuda.is_available() else False,
    dataloader_num_workers=2,                    # Async data loading
    dataloader_pin_memory=True,                  # Faster CPU->GPU transfer
    logging_steps=100,
    report_to="none",
)

trainer = FocalLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds_balanced,
    eval_dataset=val_ds,
    data_collator=data_collator,                 # Dynamic padding!
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()
trainer.save_model("model_output/best_model")
print("\n\u2705 Training complete! Best model saved to model_output/best_model")

In [ ]:
# 5. Evaluate on Test Set + Optimal Threshold Sweep
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve

test_ds = load_from_disk("data/test")
results = trainer.predict(test_ds)
logits = results.predictions
labels = results.label_ids

# --- Default threshold (0.5) ---
preds = np.argmax(logits, axis=-1)
print("=" * 50)
print("RESULTS @ threshold = 0.5")
print("=" * 50)
print(classification_report(labels, preds, target_names=["Safe", "Vulnerable"]))
print("Confusion Matrix:")
print(confusion_matrix(labels, preds))

# --- Optimal threshold sweep ---
probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
precisions, recalls, thresholds = precision_recall_curve(labels, probs)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_scores)
best_thresh = thresholds[best_idx] if best_idx < len(thresholds) else 0.5

print(f"\n{'=' * 50}")
print(f"\U0001f3af OPTIMAL THRESHOLD ANALYSIS")
print(f"{'=' * 50}")
print(f"  Best threshold: {best_thresh:.3f}")
print(f"  F1 at optimal:  {f1_scores[best_idx]:.4f}")
print(f"  Precision:      {precisions[best_idx]:.4f}")
print(f"  Recall:         {recalls[best_idx]:.4f}")

# Show results at optimal threshold
optimal_preds = (probs >= best_thresh).astype(int)
print(f"\nClassification Report @ threshold = {best_thresh:.3f}:")
print(classification_report(labels, optimal_preds, target_names=["Safe", "Vulnerable"]))
print("Confusion Matrix:")
print(confusion_matrix(labels, optimal_preds))

print(f"\n\U0001f4a1 TIP: Use threshold={best_thresh:.3f} in your inference pipeline (app.py)")

In [ ]:
# 6. Interactive Inference
from transformers import pipeline

classifier = pipeline("text-classification", model="model_output/best_model", device=0 if torch.cuda.is_available() else -1)

code = """
void vulnerable_function(char *input) {
    char buffer[10];
    strcpy(buffer, input); // Buffer overflow!
}
"""

result = classifier(code)
print(f"Prediction: {result}")

In [ ]:
# 7. Export Model to Google Drive
from google.colab import drive
drive.mount('/content/drive')
!cp -r model_output/best_model /content/drive/MyDrive/codebert_vuln_model
print("\u2705 Model saved to Google Drive!")